# <center> **Лабораторная работа №1** </center>


In [111]:
import pandas as pd


df = pd.read_csv('titanic.csv', index_col='PassengerId')
print("\nПервые 5 строк данных:")
df.head()


Первые 5 строк данных:


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### **1. Какое количество мужчин и женщин ехало на корабле?**


В качестве ответа приведите два числа через пробел.

In [112]:
sex_counts = df["Sex"].value_counts()
answer1 = f"{sex_counts['male']} {sex_counts['female']}"
print(answer1)

577 314


### **2. Какой части пассажиров удалось выжить?**

Посчитайте долю выживших пассажиров. Ответ приведите в процентах (число в интервале от 0 до 100, знак процента не нужен), округлив до двух знаков.

In [113]:
survived_count = 100 * df['Survived'].mean()
answer2 = f"{survived_count:.2f}"
print(answer2)

38.38


### **3. Какую долю пассажиры первого класса составляли среди всех пассажиров?**

Ответ приведите в процентах (число в интервале от 0 до 100, знак процента не нужен), округлив до двух знаков.

In [114]:
first_class_percent = 100 * (df['Pclass'] == 1).mean()
answer3 = f"{first_class_percent:.2f}"
print(answer3)

24.24


### **4. Какого возраста были пассажиры?**

Посчитайте среднее и медиану возраста пассажиров. В качестве ответа приведите два числа через пробел.

In [115]:
ages = df['Age'].dropna()
mean_age = ages.mean()
median_age = ages.median()
answer4 = f"{mean_age:.2f} {median_age:.2f}"
print(answer4)

29.70 28.00


### **5. Коррелируют ли число братьев/сестер с числом родителей/детей?**

Посчитайте корреляцию Пирсона между признаками SibSp и Parch.

In [116]:
correlation = df['SibSp'].corr(df['Parch'])
answer5 = f"{correlation:.2f}"
print(answer5)

0.41


### **6. Какое самое популярное женское имя на корабле?**

Извлеките из полного имени пассажира (колонка Name) его личное имя (First Name).

In [117]:
import re

def extract_female_name(full_name):
    match_bracket = re.search(r'\(([^)]+)\)', full_name)
    if match_bracket:
        inside = match_bracket.group(1).strip()
        first_name = inside.split()[0]
        first_name = first_name.strip('"\'')
        return first_name
    
    match_title = re.search(r'(?:Mrs|Miss|Ms|Mme|Lady|Mlle)\.\s+([A-Za-z\.]+)', full_name)
    if match_title:
        name = match_title.group(1).rstrip('.')
        if name and name not in ['Mr', 'Mrs', 'Miss', 'Ms']:
            return name
    
    return None

women = df[df['Sex'] == 'female'].copy()
women['FirstName'] = women['Name'].apply(extract_female_name)

women = women[women['FirstName'].notna()]

top_name = women['FirstName'].value_counts().index[0]
answer6 = top_name

print(f"{answer6}")


Mary


In [118]:
for i, ans in enumerate([answer1, answer2, answer3, answer4, answer5, answer6], 1):
    with open(f'answer{i}.txt', 'w') as f:
        f.write(str(ans))
    print(f"answer{i}.txt -> {ans}")

answer1.txt -> 577 314
answer2.txt -> 38.38
answer3.txt -> 24.24
answer4.txt -> 29.70 28.00
answer5.txt -> 0.41
answer6.txt -> Mary


# <center>**Лабораторная работа №2** </center>

In [135]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

### **1. Загрузите выборку из файла titanic.csv с помощью пакета Pandas.**

In [120]:
df = pd.read_csv('titanic.csv', index_col='PassengerId')
print("Первые 5 строк данных:")
df.head()

Первые 5 строк данных:


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### **2. Оставьте в выборке четыре признака: класс пассажира (Pclass), цену билета (Fare), возраст пассажира (Age) и его пол (Sex).**

In [121]:
features = ['Pclass', 'Fare', 'Age', 'Sex']
X = df[features].copy()
print("Первые 5 строк данных преобразованной таблицы:")
X.head()

Первые 5 строк данных преобразованной таблицы:


,Pclass,Fare,Age,Sex
PassengerId,,,,
1,3,7.2500,22.0,male
2,1,71.2833,38.0,female
3,3,7.9250,26.0,female
4,1,53.1000,35.0,female
5,3,8.0500,35.0,male


### **3. Обратите внимание, что признак Sex имеет строковые значения.**

 Преобразуем значения признака Sex в числовой формат для работы с решающим деревом.

In [122]:
X['Sex'] = X['Sex'].map({'male': 1, 'female': 0})
print("Первые 5 строк данных преобразованной таблицы:")
X.head()

Первые 5 строк данных преобразованной таблицы:


,Pclass,Fare,Age,Sex
PassengerId,,,,
1,3,7.2500,22.0,1
2,1,71.2833,38.0,0
3,3,7.9250,26.0,0
4,1,53.1000,35.0,0
5,3,8.0500,35.0,1


### **4. Выделите целевую переменную — она записана в столбце Survived.**

In [123]:
y = df['Survived'].copy()
print("Первые 10 значений целевой переменной:")
print(y.head(10))

Первые 10 значений целевой переменной:
PassengerId
1     0
2     1
3     1
4     1
5     0
6     0
7     0
8     0
9     1
10    1
Name: Survived, dtype: int64


### **5. Найдите все объекты, у которых есть пропущенные признаки, и удалите их из выборки.**

Проверяем количество пропусков в каждом признаке.

In [124]:
print("Количество пропусков (NaN) в каждом признаке:")
for col in features:
    missing = X[col].isna().sum()
    print(f"{col:10}: {missing:4} пропусков")
print(f"Всего строк в X: {len(X)}")

Количество пропусков (NaN) в каждом признаке:
Pclass    :    0 пропусков
Fare      :    0 пропусков
Age       :  177 пропусков
Sex       :    0 пропусков
Всего строк в X: 891


Удаляем строки, где есть хотя бы один пропуск.

In [125]:
print(f"До удаления пропусков: {len(X)} строк")

X_clean = X.dropna()
y_clean = y[X_clean.index]  

print(f"После удаления пропусков: {len(X_clean)} строк")
print(f"Удалено строк: {len(X) - len(X_clean)}")
print(f"Проверка: есть ли пропуски в X_clean? {X_clean.isna().any().any()}")

До удаления пропусков: 891 строк
После удаления пропусков: 714 строк
Удалено строк: 177
Проверка: есть ли пропуски в X_clean? False


### **6. Обучите решающее дерево с параметром random_state=241 и остальными параметрами по умолчанию.**

In [126]:
clf = DecisionTreeClassifier(random_state=241)
clf.fit(X_clean, y_clean)

DecisionTreeClassifier(random_state=241)

### **7. Вычислите важности признаков и найдите два признака с наибольшей важностью.**

 Их названия будут ответами для данной задачи (в качестве ответа укажите названия признаков через запятую без пробелов).

In [127]:
importances = clf.feature_importances_

print("Важности признаков:")
for name, importance in zip(features, importances):
    print(f"{name:10} : {importance:.4f}  ({importance*100:.1f}%)")

pairs = sorted(zip(features, importances), key=lambda x: x[1], reverse=True)
answer = f"{pairs[0][0]},{pairs[1][0]}"
print(f"\nДва признака с наибольшей важностью: {answer}")

with open('answer.txt', 'w') as f:
    f.write(answer)

Важности признаков:
Pclass     : 0.1400  (14.0%)
Fare       : 0.3034  (30.3%)
Age        : 0.2560  (25.6%)
Sex        : 0.3005  (30.1%)

Два признака с наибольшей важностью: Fare,Sex


# <center>**Лабораторная работа №3** </center>

### **1. Загрузите выборку Wine.**

In [129]:
wine = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"
df = pd.read_csv(wine, header=None)
print(f"\nПервые 5 строк:")
df.head()


Первые 5 строк:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


 ### **2. Извлеките из данных признаки и классы.**

In [131]:
X = df.iloc[:, 1:].values 
y = df.iloc[:, 0].values   

### 3. **Оценку качества необходимо провести методом кросс-валидации по 5 блокам (5-fold).**

Создайте генератор разбиений, который перемешивает выборку перед формированием блоков (shuffle=True). Для воспроизводимости результата, создавайте генератор KFold с фиксированным параметром random_state=42. В качестве меры качества используйте долю верных ответов (accuracy).

In [134]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

kf = KFold(n_splits=5, shuffle=True, random_state=42)
knn = KNeighborsClassifier(n_neighbors=1)
scores = cross_val_score(knn, X, y, cv=kf, scoring='accuracy')

print(f"Accuracy: {scores.mean():.4f}")

Accuracy: 0.7305
